# Decision Tree Classifier

This notebook trains and evaluates a decision tree classifier on scikit-learn's built-in Iris dataset. To use your own data, replace the data-loading cell and set `target_column`.


## 1. Imports and configuration

Install dependencies if needed: `python -m pip install pandas scikit-learn matplotlib`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

RANDOM_STATE = 42
TEST_SIZE = 0.2
MAX_DEPTH = 3  # Limit tree depth to help avoid overfitting


## 2. Load data

Iris is a small, labeled classification dataset included with scikit-learn, so no external files are required.


In [ ]:
iris = load_iris(as_frame=True)
data = iris.frame.copy()
target_column = \"target\"

# For a CSV dataset, replace the three lines above with e.g.:
# data = pd.read_csv(\"path/to/your_data.csv\")
# target_column = \"name_of_label_column\"

data.head()


In [ ]:
if target_column not in data.columns:
    raise KeyError(f\"Target column {target_column!r} not found. Available columns: {list(data.columns)}\")

X = data.drop(columns=[target_column])
y = data[target_column]

if y.isna().any():
    raise ValueError(\"The target column contains missing values; clean or remove those rows first.\")

# This starter workflow expects numeric features. Encode categorical columns before fitting.
non_numeric = X.select_dtypes(exclude=\"number\").columns.tolist()
if non_numeric:
    raise ValueError(f\"Encode non-numeric feature columns before training: {non_numeric}\")

# Simple baseline handling for numeric missing values.
X = X.fillna(X.median(numeric_only=True))
print(f\"Rows: {len(data)}, features: {X.shape[1]}, classes: {y.nunique()}\")
X.head()


## 3. Split into training and test sets

The test set is held out for a final estimate of performance. Stratification keeps class proportions similar across the two splits.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f\"Training rows: {len(X_train)} | Test rows: {len(X_test)}\")


## 4. Train the decision tree


In [ ]:
model = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=RANDOM_STATE)
model.fit(X_train, y_train)
print(\"Model trained.\")


## 5. Evaluate

Accuracy is the share of correct predictions. Review the per-class precision, recall, and F1 scores as well, especially if classes are imbalanced.


In [ ]:
predictions = model.predict(X_test)
print(f\"Test accuracy: {accuracy_score(y_test, predictions):.3f}\")
print(\"\\nClassification report:\\n\")
print(classification_report(y_test, predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap=\"Blues\")
plt.title(\"Confusion matrix ? test set\")
plt.tight_layout()
plt.show()


## 6. Inspect the trained tree


In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    model, feature_names=X.columns, class_names=[str(value) for value in model.classes_],
    filled=True, rounded=True, fontsize=9
)
plt.tight_layout()
plt.show()

feature_importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
feature_importance


## Next steps

- Replace Iris with your dataset and set the correct target column.
- Encode categorical features and consider more careful missing-value handling for real data.
- Tune `max_depth` and other tree settings using cross-validation before relying on the model.
